# Encoder Workbench

Train, evaluate, and compare per-modality encoders (WiFi / IMU / Odometry / Vision).

**What this notebook does:**
1. Load the multi-modal dataset via `FusionDataModule`
2. Inspect sample shapes, distributions, and temporal structure
3. *(later)* Train & evaluate encoders with linear probing, alignment/uniformity, etc.

In [ ]:
import sys, os
os.chdir(os.path.join(os.path.dirname(os.getcwd()), ""))  # repo root
sys.path.insert(0, ".")

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Load the dataset

In [ ]:
from src.pipeline.data import FusionDataModule

dm = FusionDataModule(
    data_dir="data/async_collection",
    train_paths=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    val_paths=[2, 13, 14],
    test_paths=[15, 16, 17],
    modalities=["imu", "odom", "wifi", "camera"],
    batch_size=64,
    normalize=True,
    wifi_pca=32,  # reduce 117 APs → 32 PCA components (set to None for raw)
)
dm.setup()
print(dm.summary())

## 2. Inspect a single sample

Each sample is anchored to a ground-truth timestamp. The dataloader returns:
- **`imu`**: `(window=32, 9)` — last ~1s of accel_xyz, gyro_xyz, roll/pitch/yaw
- **`odom`**: `(window=16, 7)` — last ~1s of odom_xy, theta, velocities, wheel speeds
- **`wifi`**: `(window=1, N)` — most recent WiFi scan (117 raw APs, or fewer if `wifi_pca` is set)
- **`camera`**: `(3, 224, 224)` — most recent RGB frame, resized + ImageNet-normalized
- **`target`**: `(2,)` — ground truth (x, y) in meters
- **`timestamp`**: scalar — simulation time in seconds

In [ ]:
sample = dm.train_ds[100]

print("Keys:", list(sample.keys()))
print()
for key in ["imu", "odom", "wifi", "camera"]:
    t = sample[key]
    print(f"{key:8s}  shape={str(list(t.shape)):16s}  dtype={t.dtype}  range=[{t.min():.3f}, {t.max():.3f}]")
print(f"\ntarget  (x, y) = ({sample['target'][0]:.2f}, {sample['target'][1]:.2f}) m")
print(f"timestamp = {sample['timestamp']:.3f} s")
print(f"path_id = {sample['path_id']}")

## 3. Inspect a batch (from DataLoader)

In [ ]:
train_loader = dm.train_dataloader()
batch = next(iter(train_loader))

print(f"Batch size: {batch['target'].shape[0]}")
print()
for key in ["imu", "odom", "wifi", "camera", "target"]:
    print(f"{key:8s}  {list(batch[key].shape)}")

## 4. Visualize modality windows

Plot what each encoder "sees" for a single sample: IMU time series, odometry trajectory, WiFi fingerprint, and camera frame.

In [ ]:
from src.pipeline.data.dataset import IMU_COLS, ODOM_COLS

sample = dm.train_ds[200]  # pick a mid-path sample

fig, axes = plt.subplots(1, 4, figsize=(22, 4))

# --- IMU window (32 steps, 9 features) ---
ax = axes[0]
imu = sample["imu"].numpy()
for i, col in enumerate(IMU_COLS[:6]):  # accel + gyro only
    ax.plot(imu[:, i], label=col, alpha=0.8)
ax.set_title("IMU window (32 steps)")
ax.set_xlabel("Step (within window)")
ax.set_ylabel("Normalized value")
ax.legend(fontsize=7, ncol=2)

# --- Odometry window (16 steps) ---
ax = axes[1]
odom = sample["odom"].numpy()
ax.plot(odom[:, 0], odom[:, 1], "o-", markersize=3)
ax.plot(odom[0, 0], odom[0, 1], "gs", markersize=8, label="start")
ax.plot(odom[-1, 0], odom[-1, 1], "r^", markersize=8, label="end")
ax.set_title("Odom window (16 steps)")
ax.set_xlabel("odom_x (normalized)")
ax.set_ylabel("odom_y (normalized)")
ax.legend()
ax.set_aspect("equal")

# --- WiFi fingerprint ---
ax = axes[2]
wifi = sample["wifi"].numpy().flatten()
ax.bar(range(len(wifi)), wifi, width=1.0, color="steelblue", alpha=0.7)
n_label = f"{len(wifi)} PCA dims" if dm.wifi_pca else f"{len(wifi)} APs"
ax.set_title(f"WiFi fingerprint ({n_label})")
ax.set_xlabel("Component" if dm.wifi_pca else "AP index")
ax.set_ylabel("Normalized value")

# --- Camera frame ---
ax = axes[3]
cam = sample["camera"].numpy()
# Undo ImageNet normalization for display
mean = np.array([0.485, 0.456, 0.406])[:, None, None]
std = np.array([0.229, 0.224, 0.225])[:, None, None]
cam_display = np.clip(cam * std + mean, 0, 1).transpose(1, 2, 0)
ax.imshow(cam_display)
ax.set_title("Camera (224x224)")
ax.axis("off")

fig.suptitle(
    f"Sample #{200} — path {sample['path_id']}, "
    f"pos=({sample['target'][0]:.1f}, {sample['target'][1]:.1f})m, "
    f"t={sample['timestamp']:.1f}s",
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.show()

## 5. Target distribution (ground truth positions)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, ds) in zip(axes, [("Train", dm.train_ds), ("Val", dm.val_ds), ("Test", dm.test_ds)]):
    targets = np.array([ds[i]["target"].numpy() for i in range(len(ds))])
    ax.scatter(targets[:, 0], targets[:, 1], s=1, alpha=0.3)
    ax.set_title(f"{name} — {len(ds)} samples")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_aspect("equal")

plt.suptitle("Ground truth positions per split", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Normalization stats

The training set stats are used for all splits (no data leakage).

In [ ]:
import pandas as pd

for mod in ["imu", "odom", "wifi"]:
    stats = dm.train_ds.stats[mod]
    if mod == "imu":
        cols = IMU_COLS
    elif mod == "odom":
        cols = ODOM_COLS
    else:
        n = len(stats["mean"])
        cols = [f"PC_{i}" for i in range(n)] if dm.wifi_pca else [f"AP_{i}" for i in range(n)]

    df = pd.DataFrame({"feature": cols, "mean": stats["mean"], "std": stats["std"]})
    print(f"\n{'='*50}")
    print(f"  {mod.upper()} normalization stats ({len(cols)} features)")
    print(f"{'='*50}")
    print(df.to_string(index=False, float_format="{:.4f}".format))

## 7. Feature correlation heatmaps

Check inter-feature correlations within each modality (helps decide encoder architecture).

In [ ]:
# Gather raw (unnormalized) data for correlation analysis
dm_raw = FusionDataModule(
    data_dir="data/async_collection",
    train_paths=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    val_paths=[2, 13, 14],
    test_paths=[15, 16, 17],
    modalities=["imu", "odom", "wifi"],  # no camera for correlation
    normalize=False,
)
dm_raw.setup()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (mod, cols) in zip(axes, [("imu", IMU_COLS), ("odom", ODOM_COLS)]):
    # Stack last-step values from N samples
    N = min(500, len(dm_raw.train_ds))
    vals = np.array([dm_raw.train_ds[i][mod][-1].numpy() for i in range(N)])
    corr = np.corrcoef(vals.T)
    sns.heatmap(corr, ax=ax, xticklabels=cols, yticklabels=cols,
                cmap="RdBu_r", vmin=-1, vmax=1, center=0, annot=True, fmt=".2f",
                annot_kws={"fontsize": 7})
    ax.set_title(f"{mod.upper()} feature correlation")

# WiFi: too many features for heatmap — show top-20 variance APs
ax = axes[2]
N = min(500, len(dm_raw.train_ds))
wifi_vals = np.array([dm_raw.train_ds[i]["wifi"].numpy().flatten() for i in range(N)])
variances = wifi_vals.var(axis=0)
top20 = np.argsort(variances)[-20:]
corr_wifi = np.corrcoef(wifi_vals[:, top20].T)
sns.heatmap(corr_wifi, ax=ax, cmap="RdBu_r", vmin=-1, vmax=1, center=0)
ax.set_title(f"WiFi top-20 variance APs correlation")

plt.tight_layout()
plt.show()

## 8. Quick sanity check — KNN baseline

Before building any encoder, check: can a simple K-nearest-neighbors regressor predict (x,y) from raw features? This sets the floor for encoder performance.

> **No camera here** — KNN on raw pixels (150K dims) is meaningless and slow. Vision needs a learned encoder (CNN/ViT) before distance-based methods work.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error

results = {}

for mod in ["imu", "odom", "wifi"]:
    # Flatten each window into a 1D feature vector
    N_train = len(dm_raw.train_ds)
    N_val = len(dm_raw.val_ds)

    X_train = np.array([dm_raw.train_ds[i][mod].numpy().flatten() for i in range(N_train)])
    y_train = np.array([dm_raw.train_ds[i]["target"].numpy() for i in range(N_train)])

    X_val = np.array([dm_raw.val_ds[i][mod].numpy().flatten() for i in range(N_val)])
    y_val = np.array([dm_raw.val_ds[i]["target"].numpy() for i in range(N_val)])

    knn = KNeighborsRegressor(n_neighbors=5, weights="distance")
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_val)

    mae = mean_absolute_error(y_val, y_pred)
    euclid = np.sqrt(((y_val - y_pred) ** 2).sum(axis=1)).mean()
    results[mod] = {"MAE": mae, "Mean Euclidean (m)": euclid}
    print(f"{mod:6s}  MAE={mae:.3f}m   Mean Euclidean={euclid:.3f}m")

print("\n--- KNN baseline (k=5, raw features, no encoder) ---")
print("This is the floor. Encoders should beat this.")

## 9. Evaluation harness — dummy encoder baseline

Run all 6 evaluation metrics on a **random linear encoder** (untrained).
This establishes the floor: any real encoder must beat these numbers.

In [ ]:
import torch.nn as nn
from src.pipeline.evaluation import (
    extract_embeddings, evaluate_encoder, print_report,
)
from src.pipeline.evaluation.encoder_eval import extract_embeddings

# --- Dummy encoder: flatten window → random linear projection → embed_dim ---
class DummyEncoder(nn.Module):
    """Random linear projection — the baseline any real encoder should beat."""
    def __init__(self, input_dim, embed_dim=64):
        super().__init__()
        self.flatten = nn.Flatten()
        self.proj = nn.Linear(input_dim, embed_dim)
    def forward(self, x):
        return self.proj(self.flatten(x))

# We need a DataModule without camera (camera is too slow for dummy baseline)
dm_eval = FusionDataModule(
    data_dir="data/async_collection",
    train_paths=[1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    val_paths=[2, 13, 14],
    test_paths=[15, 16, 17],
    modalities=["imu", "odom", "wifi"],
    batch_size=128,
    normalize=True,
    wifi_pca=32,
)
dm_eval.setup()
train_loader = dm_eval.train_dataloader()
val_loader = dm_eval.val_dataloader()

In [ ]:
# Run full evaluation on each modality with dummy encoder
from src.pipeline.data.dataset import IMU_COLS, ODOM_COLS, DEFAULT_WINDOWS

modality_input_dims = {
    "imu": DEFAULT_WINDOWS["imu"] * len(IMU_COLS),       # 32 * 9 = 288
    "odom": DEFAULT_WINDOWS["odom"] * len(ODOM_COLS),     # 16 * 7 = 112
    "wifi": DEFAULT_WINDOWS["wifi"] * 32,                 # 1 * 32 (PCA)
}

all_results = {}
for mod, in_dim in modality_input_dims.items():
    print(f"\n>>> Evaluating dummy encoder for: {mod} (input_dim={in_dim})")
    encoder = DummyEncoder(input_dim=in_dim, embed_dim=64)

    results = evaluate_encoder(
        encoder=encoder,
        train_loader=train_loader,
        val_loader=val_loader,
        modality=mod,
    )
    print_report(results, modality=mod)
    all_results[mod] = results

In [ ]:
# Summary comparison table
print(f"\n{'Modality':>10} | {'LP MAE':>8} | {'LP Euclid':>10} | {'KNN MAE':>8} | {'Align':>8} | {'Uniform':>9} | {'Eff.Dim':>8} | {'Temporal':>8}")
print("-" * 95)
for mod, res in all_results.items():
    lp = res["linear_probe"]
    kp = res["knn_probe"]
    au = res["alignment_uniformity"]
    ed = res["effective_dimensionality"]
    ts = res["temporal_smoothness"]
    print(f"{mod:>10} | {lp['mae']:>7.3f}m | {lp['mean_euclidean']:>9.3f}m | {kp['mae']:>7.3f}m | {au['alignment']:>8.4f} | {au['uniformity']:>9.4f} | {ed['participation_ratio']:>7.1f}/{ed['embed_dim']} | {ts['correlation']:>8.3f}")

print("\nThese are RANDOM BASELINE numbers. Any trained encoder must beat them.")

## Next steps

- Implement real per-modality encoders in `src/pipeline/encoders/`
- Re-run Section 9 with trained encoders — compare against the random baseline above
- Camera encoder evaluation will be added once a vision encoder (ViT/ResNet) is implemented